# Fake News Detection - 03 TF-IDF + SVM

In [ ]:
import os
import sys

sys.path.insert(0, '../src')

import gc
import polars as pl

from data import standard_labels, binary_labels, label_distribution, split_data
from evaluation import format_test_results, qualitative_test, report, type_summary
from features import save_vectorizer, tfidf_vectorizer, transform_in_chunks
from models import save_model, train_svm
from preprocessing import preprocess_text

os.makedirs('../artifacts/models', exist_ok=True)
os.makedirs('../artifacts/vectorizers', exist_ok=True)
os.makedirs('../artifacts/results', exist_ok=True)

In [ ]:
df = pl.read_csv('../data/preprocessed/995,000_rows_preprocessed.csv', columns=['type', 'content'])
df = standard_labels(df)
df.head()

In [ ]:
df_train, df_val, df_test = split_data(df)

print('Train split:')
print(label_distribution(df_train))
print('\nVal split:')
print(label_distribution(df_val))
print('\nTest split:')
print(label_distribution(df_test))

In [ ]:
vectorizer = tfidf_vectorizer()
sample_series = df_train['content'].sample(fraction=0.3, seed=1)
vectorizer.fit(sample_series)
del sample_series
gc.collect()

In [ ]:
y_train = binary_labels(df_train)
y_val = binary_labels(df_val)
y_test = binary_labels(df_test)
val_types = df_val['type'].to_numpy()
val_texts = df_val['content'].to_list()

tfidf_train = transform_in_chunks(df_train['content'], vectorizer)
del df_train
gc.collect()

tfidf_val = transform_in_chunks(df_val['content'], vectorizer)
del df_val
gc.collect()

tfidf_test = transform_in_chunks(df_test['content'], vectorizer)
del df_test
gc.collect()

In [ ]:
svm = train_svm(tfidf_train, y_train, c=0.3727593720314942)
y_val_pred = svm.predict(tfidf_val)
y_val_score = svm.decision_function(tfidf_val)
summary = report(y_val, y_val_pred)
print(summary)

In [ ]:
type_metrics = type_summary(val_types, y_val_pred, y_val_score)
with pl.Config(tbl_rows=-1):
    print(type_metrics)

In [ ]:
save_model(svm, '../artifacts/models/03_svm_model.joblib')
save_vectorizer(vectorizer, '../artifacts/vectorizers/03_svm_vectorizer.joblib')
with open('../artifacts/results/03_svm_validation_report.txt', 'w', encoding='utf-8') as file:
    file.write(summary)
with open('../artifacts/results/03_svm_validation_by_type.txt', 'w', encoding='utf-8') as file:
    file.write(str(type_metrics))

In [ ]:
qualitative_rows = qualitative_test(
    '../news/test_articles.txt',
    model=svm,
    vectorizer=vectorizer,
    preprocess_text=preprocess_text,
)
print(format_test_results(qualitative_rows))